# LangChain and LangGraph: Step-by-Step Foundations

A practical, beginner-friendly tutorial for building LLM applications, RAG workflows,
tools, and stateful agent workflows.

This notebook is intentionally incremental. Each step introduces **one new idea**
and then immediately demonstrates it with a small code example.

---

## Learning outcomes

By the end of this notebook, you should be able to:

- explain what LangChain and LangGraph are used for;
- build a simple LangChain pipeline with prompts, models, and parsers;
- add normal Python functions to a LangChain pipeline;
- create documents, chunks, embeddings, a Chroma vector store, and a retriever;
- build a simple Retrieval-Augmented Generation workflow;
- define and test LangChain tools;
- create a LangGraph `State`, nodes, edges, branches, and loops;
- place an LLM, a LangChain tool, and a RAG workflow inside LangGraph nodes;
- build a small Agentic RAG workflow with retrieval, evaluation, retry, and generation.

---

## Conceptual map

```text
LANGCHAIN
Model → Prompt → Parser → Documents → Embeddings → Retriever → RAG → Tools

LANGGRAPH
State → Nodes → Edges → Conditions → Loops → Tools → RAG → Agentic RAG
```

The examples use a small local Qwen model so the core tutorial can run without a
paid LLM API.

# 0. Environment Setup

### Goal
Prepare a clean Python environment with the libraries used throughout the notebook.

### Why this matters
AI libraries evolve quickly and often depend on specific package versions.
A virtual environment keeps this project isolated from other Python projects.

### Local Jupyter setup

**Windows PowerShell**

```powershell
python -m venv .venv
.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
```

**macOS / Linux**

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
```

Then install the required packages.

### Google Colab
Colab already provides an isolated runtime, so a second virtual environment is
usually unnecessary. Run the next cell directly.

In [ ]:
%pip install -q -U     langchain     langgraph     langchain-huggingface     langchain-chroma     langchain-text-splitters     sentence-transformers     transformers     accelerate     chromadb

## Step 1 — Check the Runtime

**Goal.** Confirm whether a GPU is available.

**Why it matters.** Local language models run much faster on a GPU. The notebook
can still run on CPU, but generation will be slower.

**What to notice.** If `CUDA available` is `True`, the next line should show the
GPU name.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU.")

# Part I — LangChain Foundations

LangChain helps us connect reusable components such as:

```text
Model
Prompt
Output parser
Documents
Text splitter
Embeddings
Vector store
Retriever
Tools
```

A useful mental model is:

> **LangChain organizes capabilities.**

We will begin with the smallest possible model call and gradually build a chain.

## Step 2 — Load a Local Qwen Model

**Goal.** Create the language-model component used in later examples.

**Why it matters.** LangChain is not itself a language model. It provides a common
way to connect a model to the rest of an application.

We use `Qwen/Qwen3-0.6B` because it is small enough for an introductory Colab lab.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

model_id = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

qwen_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

text_pipe = pipeline(
    "text-generation",
    model=qwen_model,
    tokenizer=tokenizer,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.6,
    top_p=0.85,
    return_full_text=False
)

model = HuggingFacePipeline(
    pipeline=text_pipe
)

print("Model ready:", model_id)

## Step 3 — Call the Model Directly

**Goal.** Understand the simplest model interaction before adding LangChain
components around it.

The flow is:

```text
Prompt text
   ↓
Model
   ↓
Generated text
```

This direct call is useful for quick tests and helps us see exactly what the model
receives.

In [ ]:
response = model.invoke(
    "Explain machine learning in one short sentence."
)

print(response)

## Step 4 — Create a Prompt Template

**Goal.** Replace a hard-coded prompt with a reusable template.

A hard-coded prompt:

```text
Explain neural networks in simple language.
```

A reusable template:

```text
Explain {topic} in simple language.
```

**Why it matters.** Templates separate fixed instructions from changing input.
This makes prompts easier to reuse, test, and maintain.

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "Explain {topic} in simple language."
)

formatted_prompt = prompt.format(
    topic="neural networks"
)

print(formatted_prompt)

## Step 5 — Connect the Prompt to the Model Manually

**Goal.** See the two operations separately before creating a chain.

```text
Input value
   ↓
PromptTemplate
   ↓
Formatted prompt
   ↓
Model
```

This manual version makes the next step easier to understand.

In [ ]:
formatted_prompt = prompt.format(
    topic="vector databases"
)

response = model.invoke(
    formatted_prompt
)

print(response)

## Step 6 — Create the First LangChain Chain

**Goal.** Connect the prompt and model using the pipe operator.

```python
prompt | model
```

The `|` means:

> Send the output of the component on the left into the component on the right.

So the chain becomes:

```text
Input dictionary
      ↓
PromptTemplate
      ↓
Model
```

In [ ]:
chain = prompt | model

response = chain.invoke({
    "topic": "retrieval augmented generation"
})

print(response)

## Step 7 — Add an Output Parser

**Goal.** Add a final component that returns a clean Python string.

The chain now becomes:

```text
Prompt
  ↓
Model
  ↓
StrOutputParser
```

**Why it matters.** Predictable output types make later code easier to work with.
Parsers are especially useful when an application expects a specific structure.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

chain = prompt | model | parser

response = chain.invoke({
    "topic": "embeddings"
})

print(response)
print("Output type:", type(response))

## Step 8 — Create a Small Python Function

**Goal.** Add ordinary Python logic to the workflow.

Suppose we want to clean a topic before placing it into the prompt.

```python
def clean_topic(text):
    return text.strip().lower()
```

Small helper functions are useful for preprocessing, validation, formatting, and
business rules.

In [ ]:
def clean_topic(text):
    return text.strip().lower()

print(
    clean_topic("   VECTOR DATABASES   ")
)

## Step 9 — Turn the Function into a Runnable

**Goal.** Make a normal Python function compatible with a LangChain pipeline.

`RunnableLambda` wraps a Python function so it can participate in a chain.

```text
Raw input
   ↓
Python function
   ↓
Next LangChain component
```

In [ ]:
from langchain_core.runnables import RunnableLambda

clean_step = RunnableLambda(
    clean_topic
)

print(
    clean_step.invoke("   VECTOR DATABASES   ")
)

## Step 10 — Use the Small Function in a Chain

**Goal.** Combine ordinary Python logic with LangChain components.

The workflow is now:

```text
Raw text
   ↓
clean_topic()
   ↓
Convert to {"topic": ...}
   ↓
PromptTemplate
   ↓
Model
   ↓
Parser
```

This pattern is important because real applications often mix LLM components with
normal Python functions.

In [ ]:
topic_prompt = PromptTemplate.from_template(
    "Give a beginner-friendly definition of {topic}."
)

small_chain = (
    clean_step
    | (lambda topic: {"topic": topic})
    | topic_prompt
    | model
    | parser
)

result = small_chain.invoke(
    "   VECTOR DATABASES   "
)

print(result)

## Step 11 — Use Multiple Prompt Variables

**Goal.** Build a prompt that accepts more than one input.

```text
Explain {topic} for a {audience}.
```

**Why it matters.** Real prompts often combine several pieces of information such
as topic, audience, tone, format, or task.

In [ ]:
audience_prompt = PromptTemplate.from_template(
    "Explain {topic} for a {audience}."
)

audience_chain = (
    audience_prompt
    | model
    | parser
)

result = audience_chain.invoke({
    "topic": "RAG",
    "audience": "beginner"
})

print(result)

# Part II — LangChain for RAG

RAG stands for **Retrieval-Augmented Generation**.

Instead of asking the model to answer only from its training, we first retrieve
relevant information and place that information into the prompt.

```text
Question
   ↓
Retriever
   ↓
Relevant context
   ↓
Model
   ↓
Grounded answer
```

## Step 12 — Create a LangChain `Document`

**Goal.** Store text together with metadata.

A `Document` contains:

```text
page_content
metadata
```

**Why it matters.** When a chunk is retrieved later, metadata helps us identify
its source.

In [ ]:
from langchain_core.documents import Document

doc = Document(
    page_content="Opened electronics may be returned within 14 days.",
    metadata={
        "source": "return_policy.txt"
    }
)

print(doc.page_content)
print(doc.metadata)

## Step 13 — Create a Small Knowledge Base

**Goal.** Build a few documents covering different topics.

Keeping the collection small at first makes retrieval behavior easy to inspect.

In [ ]:
documents = [
    Document(
        page_content=(
            "Most unopened products may be returned within 30 days. "
            "Opened electronics may be returned within 14 days."
        ),
        metadata={"source": "returns.txt"}
    ),

    Document(
        page_content=(
            "Standard shipping costs $7.99. "
            "Standard shipping is free when the subtotal is at least $75."
        ),
        metadata={"source": "shipping.txt"}
    ),

    Document(
        page_content=(
            "Student discounts are 10 percent on eligible accessories. "
            "Percentage discounts cannot be stacked."
        ),
        metadata={"source": "discounts.txt"}
    )
]

print("Number of documents:", len(documents))

## Step 14 — Split Documents into Chunks

**Goal.** Break longer documents into smaller searchable pieces.

Two important parameters:

- `chunk_size`: approximate maximum chunk length;
- `chunk_overlap`: repeated text between neighboring chunks.

**Why it matters.** Smaller chunks often improve retrieval precision, while overlap
helps preserve context near boundaries.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=20
)

chunks = splitter.split_documents(
    documents
)

print("Chunks:", len(chunks))

for chunk in chunks:
    print("\nSource:", chunk.metadata["source"])
    print(chunk.page_content)

## Step 15 — Create Embeddings

**Goal.** Convert text into numeric vectors representing semantic meaning.

```text
Text
  ↓
Embedding model
  ↓
Vector
```

**Why it matters.** Embeddings let us search by meaning instead of relying only on
exact keyword matches.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={
        "device": "cuda" if torch.cuda.is_available() else "cpu"
    }
)

vector = embeddings.embed_query(
    "return an opened product"
)

print("Vector length:", len(vector))
print("First five values:", vector[:5])

## Step 16 — Store the Chunks in Chroma

**Goal.** Create a vector store that can perform similarity search.

Chroma stores the vectors created from our document chunks.

```text
Chunks
   ↓
Embeddings
   ↓
Chroma
```

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="tutorial_knowledge"
)

print("Chroma vector store ready.")

## Step 17 — Perform Similarity Search

**Goal.** Retrieve the document chunks most similar to a question.

`k=2` means:

> Return the two best matching chunks.

Notice that the user question does not have to exactly match the document wording.

In [ ]:
results = vectorstore.similarity_search(
    "How long can I return an opened electronic item?",
    k=2
)

for result in results:
    print("\nSource:", result.metadata["source"])
    print(result.page_content)

## Step 18 — Create a Retriever

**Goal.** Use a standard retrieval interface.

```text
Question
   ↓
Retriever
   ↓
Relevant Documents
```

**Why it matters.** The rest of the application can work with a retriever without
needing to know the details of the vector-store implementation.

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

retrieved_docs = retriever.invoke(
    "When is standard shipping free?"
)

for doc in retrieved_docs:
    print(doc.page_content)

## Step 19 — Create a Small Formatting Function

**Goal.** Convert retrieved `Document` objects into one text block for the model.

The retriever returns a list of documents. The model needs prompt text.

```text
List[Document]
     ↓
format_docs()
     ↓
Context string
```

In [ ]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

context = format_docs(
    retrieved_docs
)

print(context)

## Step 20 — Build RAG Manually

**Goal.** See every stage before wrapping the process in a reusable function.

```text
Question
   ↓
Retriever
   ↓
format_docs()
   ↓
Prompt
   ↓
Model
```

This manual version is useful for understanding and debugging.

In [ ]:
question = "Can I combine two percentage discounts?"

docs = retriever.invoke(
    question
)

context = format_docs(
    docs
)

rag_prompt = f"""
Answer the question using only the context below.

QUESTION:
{question}

CONTEXT:
{context}
"""

answer = model.invoke(
    rag_prompt
)

print(answer)

## Step 21 — Wrap RAG in a Small Function

**Goal.** Turn the manual RAG steps into a reusable function.

Once the workflow is understood, a function keeps later code cleaner.

In [ ]:
def rag(question):
    docs = retriever.invoke(
        question
    )

    context = format_docs(
        docs
    )

    prompt = f"""
Answer using only the context.

QUESTION:
{question}

CONTEXT:
{context}
"""

    return model.invoke(
        prompt
    )

## Step 22 — Test the RAG Function

**What to notice.** The answer is generated by the language model, but the factual
content comes from retrieved documents.

In [ ]:
print(
    rag(
        "Can student percentage discounts be stacked?"
    )
)

# Part III — LangChain Tools

A tool is a function exposed as a capability that an agent or workflow can use.

Tools are useful when a task should be handled by reliable code instead of by
free-form language generation.

Examples include:

```text
Calculator
Database query
API call
Document search
File lookup
```

## Step 23 — Create the First Tool

**Goal.** Convert a small Python function into a LangChain tool.

The docstring is important because it describes what the tool does.

In [ ]:
from langchain_core.tools import tool

@tool
def multiply_numbers(a: float, b: float) -> float:
    """Multiply two numeric values."""
    return a * b

## Step 24 — Test the Tool Directly

**Why test tools first?** A tool should work correctly before it is placed inside
an agent or graph. This makes debugging much easier.

In [ ]:
result = multiply_numbers.invoke({
    "a": 23.5,
    "b": 18
})

print(result)

## Step 25 — Add Business Calculation Tools

**Goal.** Create focused tools that each perform one deterministic calculation.

Small, focused tools are easier to understand, test, and route correctly.

In [ ]:
@tool
def calculate_discount(
    price: float,
    discount_percent: float
) -> float:
    """Calculate the final price after a percentage discount."""

    discount = price * discount_percent / 100
    return price - discount


@tool
def calculate_profit(
    revenue: float,
    cost: float
) -> float:
    """Calculate profit from revenue and total cost."""

    return revenue - cost

## Step 26 — Test the Calculation Tools

In [ ]:
print(
    calculate_discount.invoke({
        "price": 120,
        "discount_percent": 25
    })
)

print(
    calculate_profit.invoke({
        "revenue": 1500,
        "cost": 900
    })
)

# LangChain Checkpoint

At this point, we have built the following pieces:

```text
Model
Prompt
Parser
Python function in a chain
Documents
Chunking
Embeddings
Chroma
Retriever
RAG
Tools
```

These are capabilities. The next question is:

> How should those capabilities execute when a workflow has multiple steps,
> branches, retries, or loops?

That is where LangGraph becomes useful.

# Part IV — LangGraph Foundations

LangGraph helps us explicitly control the execution of a stateful workflow.

A useful mental model is:

> **LangGraph organizes execution.**

We will again start with the smallest possible example.

## Step 27 — Define State

**Goal.** Define the information that moves through the graph.

State is shared data. Nodes read it and return updates to it.

We begin with one field:

```text
message
```

In [ ]:
from typing import TypedDict

class State(TypedDict):
    message: str

## Step 28 — Inspect State as Normal Python Data

**Goal.** Reinforce that state is simply structured data.

Before building a graph, we can use the dictionary normally.

In [ ]:
state = {
    "message": "Hello"
}

print(state["message"])

## Step 29 — Create the First Node

**Goal.** Create a function that reads state and returns an update.

A node is usually just a Python function.

```text
State before:
message = "Hello"

Node:
add_text()

State after:
message = "Hello World"
```

In [ ]:
def add_text(state: State):
    return {
        "message": state["message"] + " World"
    }

## Step 30 — Test the Node Before Building the Graph

**Why this matters.** Small functions are easier to debug outside the graph.
Always verify node logic independently when possible.

In [ ]:
print(
    add_text({
        "message": "Hello"
    })
)

## Step 31 — Create a `StateGraph`

**Goal.** Create an empty graph that uses our state schema.

At this point, the graph has no nodes and no edges.

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(
    State
)

## Step 32 — Add the Node

**Goal.** Register the Python function as a named graph node.

```text
Node name: add_text
Function:  add_text()
```

In [ ]:
builder.add_node(
    "add_text",
    add_text
)

## Step 33 — Connect `START`

**Goal.** Define the graph's entry point.

```text
START
  ↓
add_text
```

In [ ]:
builder.add_edge(
    START,
    "add_text"
)

## Step 34 — Connect `END`

**Goal.** Define where the workflow stops.

```text
START
  ↓
add_text
  ↓
END
```

In [ ]:
builder.add_edge(
    "add_text",
    END
)

## Step 35 — Compile the Graph

**Goal.** Convert the graph definition into a runnable graph.

The builder describes the graph. `compile()` prepares it for execution.

In [ ]:
graph = builder.compile()

## Step 36 — Invoke the Graph

**Goal.** Run the graph with an initial state.

Data flow:

```text
{"message": "Hello"}
        ↓
START
        ↓
add_text
        ↓
{"message": "Hello World"}
        ↓
END
```

In [ ]:
result = graph.invoke({
    "message": "Hello"
})

print(result)

## Step 37 — Add a Second Node

**Goal.** Pass updated state through two small functions.

```text
START
  ↓
add_world
  ↓
add_exclamation
  ↓
END
```

The second node receives the state after the first node updates it.

In [ ]:
class MessageState(TypedDict):
    message: str


def add_world(state: MessageState):
    return {
        "message": state["message"] + " World"
    }


def add_exclamation(state: MessageState):
    return {
        "message": state["message"] + "!"
    }

## Step 38 — Build the Two-Node Graph

**Key idea.**

- Nodes define **what work happens**.
- Edges define **what happens next**.

In [ ]:
builder2 = StateGraph(
    MessageState
)

builder2.add_node(
    "add_world",
    add_world
)

builder2.add_node(
    "add_exclamation",
    add_exclamation
)

builder2.add_edge(
    START,
    "add_world"
)

builder2.add_edge(
    "add_world",
    "add_exclamation"
)

builder2.add_edge(
    "add_exclamation",
    END
)

graph2 = builder2.compile()

## Step 39 — Run the Two-Node Graph

Expected state progression:

```text
Hello
  ↓
Hello World
  ↓
Hello World!
```

In [ ]:
print(
    graph2.invoke({
        "message": "Hello"
    })
)

## Step 40 — Use Multiple State Fields

**Goal.** Show that state can carry several values.

Real applications may store:

```text
question
answer
route
score
documents
attempt_count
```

A node only needs to return the fields it changes.

In [ ]:
class MultiState(TypedDict):
    name: str
    message: str
    count: int


def create_message(state: MultiState):
    return {
        "message": f"Hello {state['name']}"
    }


def increase_count(state: MultiState):
    return {
        "count": state["count"] + 1
    }

## Step 41 — Build the Multi-Field Graph

`create_message()` changes `message`.

`increase_count()` changes `count`.

The other state values remain available.

In [ ]:
builder3 = StateGraph(
    MultiState
)

builder3.add_node(
    "create_message",
    create_message
)

builder3.add_node(
    "increase_count",
    increase_count
)

builder3.add_edge(
    START,
    "create_message"
)

builder3.add_edge(
    "create_message",
    "increase_count"
)

builder3.add_edge(
    "increase_count",
    END
)

graph3 = builder3.compile()

## Step 42 — Run the Multi-Field Graph

In [ ]:
print(
    graph3.invoke({
        "name": "Alex",
        "message": "",
        "count": 0
    })
)

# Part V — Branches and Loops

Fixed edges are useful when every run follows the same path.

Agents and adaptive workflows often need decisions.

LangGraph supports this with **conditional edges**.

## Step 43 — Create a Conditional Routing Function

**Goal.** Choose between two paths based on state.

Example:

```text
score
  ↓
decision
 /     \
pass   fail
```

In [ ]:
class ScoreState(TypedDict):
    score: int
    result: str


def pass_node(state: ScoreState):
    return {
        "result": "Pass"
    }


def fail_node(state: ScoreState):
    return {
        "result": "Fail"
    }


def decide_result(state: ScoreState):
    if state["score"] >= 60:
        return "pass_node"

    return "fail_node"

## Step 44 — Build the Conditional Graph

`add_conditional_edges()` runs the routing function and uses its return value to
select the next node.

In [ ]:
score_builder = StateGraph(
    ScoreState
)

score_builder.add_node(
    "pass_node",
    pass_node
)

score_builder.add_node(
    "fail_node",
    fail_node
)

score_builder.add_conditional_edges(
    START,
    decide_result
)

score_builder.add_edge(
    "pass_node",
    END
)

score_builder.add_edge(
    "fail_node",
    END
)

score_graph = score_builder.compile()

## Step 45 — Test Both Branches

A useful graph-testing habit is to deliberately test at least one input for every
possible branch.

In [ ]:
print(
    score_graph.invoke({
        "score": 85,
        "result": ""
    })
)

print(
    score_graph.invoke({
        "score": 45,
        "result": ""
    })
)

## Step 46 — Route After a Node Updates State

**Goal.** Use a more realistic pattern:

```text
START
  ↓
check_score
  ↓
updated state
  ↓
route
 /   \
pass fail
```

This pattern is important because later we will retrieve information, evaluate it,
and then choose the next step.

In [ ]:
class DecisionState(TypedDict):
    score: int
    status: str


def check_score(state: DecisionState):
    if state["score"] >= 60:
        return {
            "status": "passing"
        }

    return {
        "status": "failing"
    }


def route_status(state: DecisionState):
    if state["status"] == "passing":
        return "passed"

    return "failed"


def passed(state: DecisionState):
    return {
        "status": "Student passed"
    }


def failed(state: DecisionState):
    return {
        "status": "Student failed"
    }

## Step 47 — Build the Decision Graph

In [ ]:
decision_builder = StateGraph(
    DecisionState
)

decision_builder.add_node(
    "check_score",
    check_score
)

decision_builder.add_node(
    "passed",
    passed
)

decision_builder.add_node(
    "failed",
    failed
)

decision_builder.add_edge(
    START,
    "check_score"
)

decision_builder.add_conditional_edges(
    "check_score",
    route_status
)

decision_builder.add_edge(
    "passed",
    END
)

decision_builder.add_edge(
    "failed",
    END
)

decision_graph = decision_builder.compile()

## Step 48 — Run the Decision Graph

In [ ]:
print(
    decision_graph.invoke({
        "score": 72,
        "status": ""
    })
)

## Step 49 — Create a Loop

**Goal.** Allow the graph to repeat a node.

```text
increase
   ↓
count < 3?
 /       \
yes      no
 |        |
 └ repeat END
```

**Why it matters.** Agentic systems often retry retrieval, validation, planning,
or tool use.

In [ ]:
class CounterState(TypedDict):
    count: int


def increase(state: CounterState):
    return {
        "count": state["count"] + 1
    }


def should_continue(state: CounterState):
    if state["count"] < 3:
        return "increase"

    return END

## Step 50 — Build the Loop

The conditional edge either sends execution back to `increase` or stops at `END`.

In [ ]:
counter_builder = StateGraph(
    CounterState
)

counter_builder.add_node(
    "increase",
    increase
)

counter_builder.add_edge(
    START,
    "increase"
)

counter_builder.add_conditional_edges(
    "increase",
    should_continue
)

counter_graph = counter_builder.compile()

## Step 51 — Run the Loop

Expected result:

```text
0 → 1 → 2 → 3 → END
```

In [ ]:
print(
    counter_graph.invoke({
        "count": 0
    })
)

## Step 52 — Add a Safety Boundary

**Goal.** Store a maximum step count in state.

Every loop should have a clear stopping rule.

This becomes especially important when each iteration may call an LLM, API, or
retriever.

In [ ]:
class SafeState(TypedDict):
    count: int
    max_steps: int


def safe_increase(state: SafeState):
    return {
        "count": state["count"] + 1
    }


def safe_continue(state: SafeState):
    if state["count"] < state["max_steps"]:
        return "increase"

    return END

## Step 53 — Build and Run the Bounded Loop

In [ ]:
safe_builder = StateGraph(
    SafeState
)

safe_builder.add_node(
    "increase",
    safe_increase
)

safe_builder.add_edge(
    START,
    "increase"
)

safe_builder.add_conditional_edges(
    "increase",
    safe_continue
)

safe_graph = safe_builder.compile()

print(
    safe_graph.invoke({
        "count": 0,
        "max_steps": 4
    })
)

# Part VI — Put LangChain Components Inside LangGraph

LangGraph controls execution.

A node can call:

```text
an LLM
a LangChain tool
a retriever
a database
an API
another graph
```

This is where LangChain capabilities and LangGraph orchestration begin to work
together.

## Step 54 — Put the LLM Inside a Node

**Goal.** Create the smallest LangGraph workflow that calls Qwen.

```text
START
  ↓
ask_model
  ↓
END
```

LangGraph is not the model. It controls when the node that calls the model runs.

In [ ]:
class LLMState(TypedDict):
    question: str
    answer: str


def ask_model(state: LLMState):
    response = model.invoke(
        state["question"]
    )

    return {
        "answer": response
    }

## Step 55 — Build the LLM Graph

In [ ]:
llm_builder = StateGraph(
    LLMState
)

llm_builder.add_node(
    "ask_model",
    ask_model
)

llm_builder.add_edge(
    START,
    "ask_model"
)

llm_builder.add_edge(
    "ask_model",
    END
)

llm_graph = llm_builder.compile()

## Step 56 — Run the LLM Graph

In [ ]:
result = llm_graph.invoke({
    "question": "Explain RAG in one sentence.",
    "answer": ""
})

print(result["answer"])

## Step 57 — Put a LangChain Tool Inside a Node

**Goal.** Call the `calculate_profit` tool from a LangGraph node.

```text
LangGraph node
     ↓
LangChain tool
     ↓
Python calculation
```

In [ ]:
class ProfitState(TypedDict):
    revenue: float
    cost: float
    answer: str


def profit_tool_node(state: ProfitState):
    result = calculate_profit.invoke({
        "revenue": state["revenue"],
        "cost": state["cost"]
    })

    return {
        "answer": str(result)
    }

## Step 58 — Build and Run the Tool Graph

In [ ]:
profit_builder = StateGraph(
    ProfitState
)

profit_builder.add_node(
    "calculate_profit",
    profit_tool_node
)

profit_builder.add_edge(
    START,
    "calculate_profit"
)

profit_builder.add_edge(
    "calculate_profit",
    END
)

profit_graph = profit_builder.compile()

print(
    profit_graph.invoke({
        "revenue": 1500,
        "cost": 900,
        "answer": ""
    })
)

# Part VII — Simple Routing Between Capabilities

We now create a small assistant that chooses between:

```text
Profit calculation
General LLM response
```

This is the beginning of agent-like behavior: the workflow does not always follow
the same path.

## Step 59 — Create a Router Node

**Goal.** Inspect the question and store a route in state.

For teaching, we use a transparent Python rule first.

In [ ]:
class RouterState(TypedDict, total=False):
    question: str
    route: str
    answer: str


def simple_router(state: RouterState):
    if "profit" in state["question"].lower():
        return {
            "route": "profit"
        }

    return {
        "route": "general"
    }

## Step 60 — Create Specialist Nodes

One node uses a calculation tool.

The other uses the LLM.

In [ ]:
import re

def router_profit_node(state: RouterState):
    numbers = [
        float(x)
        for x in re.findall(
            r"\d+(?:\.\d+)?",
            state["question"].replace(",", "")
        )
    ]

    if len(numbers) < 2:
        return {
            "answer": "Please provide revenue and cost."
        }

    result = calculate_profit.invoke({
        "revenue": numbers[0],
        "cost": numbers[1]
    })

    return {
        "answer": str(result)
    }


def router_general_node(state: RouterState):
    return {
        "answer": model.invoke(
            state["question"]
        )
    }

## Step 61 — Create the Routing Function

The router node stores either:

```text
profit
```

or:

```text
general
```

This function maps that value to a graph node.

In [ ]:
def choose_next_node(state: RouterState):
    if state["route"] == "profit":
        return "profit_node"

    return "general_node"

## Step 62 — Build the Router Graph

In [ ]:
router_builder = StateGraph(
    RouterState
)

router_builder.add_node(
    "router",
    simple_router
)

router_builder.add_node(
    "profit_node",
    router_profit_node
)

router_builder.add_node(
    "general_node",
    router_general_node
)

router_builder.add_edge(
    START,
    "router"
)

router_builder.add_conditional_edges(
    "router",
    choose_next_node
)

router_builder.add_edge(
    "profit_node",
    END
)

router_builder.add_edge(
    "general_node",
    END
)

router_graph = router_builder.compile()

## Step 63 — Test Both Routes

In [ ]:
print(
    router_graph.invoke({
        "question": "Revenue is 1500 and cost is 900. What is the profit?"
    })
)

print()

print(
    router_graph.invoke({
        "question": "Explain embeddings in one sentence."
    })
)

# Part VIII — Basic RAG in LangGraph

We already built RAG as a normal Python function.

Now we represent the same process as graph nodes:

```text
START
  ↓
retrieve
  ↓
generate
  ↓
END
```

This makes it easier to add branching and retries later.

## Step 64 — Define RAG State

The graph needs to carry:

```text
question
context
answer
```

In [ ]:
class RAGState(TypedDict, total=False):
    question: str
    context: str
    answer: str

## Step 65 — Create the Retrieval Node

**Goal.** Retrieve relevant documents and store formatted context in state.

In [ ]:
def retrieve_node(state: RAGState):
    docs = retriever.invoke(
        state["question"]
    )

    return {
        "context": format_docs(docs)
    }

## Step 66 — Create the Generation Node

**Goal.** Use the retrieved context to generate the final answer.

In [ ]:
def generate_node(state: RAGState):
    prompt = f"""
Answer using only the context below.

QUESTION:
{state['question']}

CONTEXT:
{state['context']}
"""

    return {
        "answer": model.invoke(
            prompt
        )
    }

## Step 67 — Build the Basic RAG Graph

In [ ]:
rag_builder = StateGraph(
    RAGState
)

rag_builder.add_node(
    "retrieve",
    retrieve_node
)

rag_builder.add_node(
    "generate",
    generate_node
)

rag_builder.add_edge(
    START,
    "retrieve"
)

rag_builder.add_edge(
    "retrieve",
    "generate"
)

rag_builder.add_edge(
    "generate",
    END
)

rag_graph = rag_builder.compile()

## Step 68 — Run the Basic RAG Graph

In [ ]:
result = rag_graph.invoke({
    "question": "When is standard shipping free?"
})

print(result["answer"])

# Part IX — Small Agentic RAG Workflow

Basic RAG always follows the same path:

```text
retrieve → generate
```

Agentic RAG adds a decision:

```text
retrieve
   ↓
evaluate evidence
   ↓
good enough?
 /          \
yes          no
 ↓            ↓
generate    rewrite query
                ↓
             retrieve again
```

This introduces the two key ideas that make the workflow more agentic:

- **decision-making**
- **controlled iteration**

## Step 69 — Define Agentic RAG State

We need a few additional fields:

- `query`: the current retrieval query;
- `attempts`: number of retrieval attempts;
- `score`: simple evidence-quality score.

In [ ]:
class AgenticRAGState(TypedDict, total=False):
    question: str
    query: str
    context: str
    attempts: int
    score: float
    answer: str

## Step 70 — Create the Retrieval Node with an Attempt Counter

Each retrieval increments `attempts`.

That value gives us a clear safety boundary for retries.

In [ ]:
def agentic_retrieve(state: AgenticRAGState):
    query = state.get(
        "query",
        state["question"]
    )

    docs = retriever.invoke(
        query
    )

    return {
        "context": format_docs(docs),
        "attempts": state.get("attempts", 0) + 1
    }

## Step 71 — Create a Simple Evidence Checker

For teaching, we use a transparent word-overlap score.

This is intentionally simple.

A production system might instead use:

- an LLM-based grader;
- a reranker;
- retrieval scores;
- a learned relevance model.

In [ ]:
def check_evidence(state: AgenticRAGState):
    question_words = set(
        state["question"].lower().split()
    )

    context_words = set(
        state.get("context", "").lower().split()
    )

    overlap = question_words & context_words

    score = (
        len(overlap) / len(question_words)
        if question_words
        else 0
    )

    return {
        "score": score
    }

## Step 72 — Create the Decision Function

Rules:

```text
If evidence looks useful:
    generate

If evidence is weak and we still have a retry:
    rewrite

Otherwise:
    generate with the best evidence available
```

The maximum attempt count prevents an infinite loop.

In [ ]:
def decide_rag(state: AgenticRAGState):
    if state.get("score", 0) >= 0.2:
        return "generate"

    if state.get("attempts", 0) < 2:
        return "rewrite"

    return "generate"

## Step 73 — Create the Query-Rewrite Node

**Goal.** Ask the model to create a shorter search-oriented version of the
original question.

Notice that we update `query`, not `question`.

The original user question remains unchanged.

In [ ]:
def rewrite_query(state: AgenticRAGState):
    prompt = f"""
Rewrite this question as a short search query.

Question:
{state['question']}
"""

    new_query = model.invoke(
        prompt
    ).strip().split("\n")[0]

    return {
        "query": new_query
    }

## Step 74 — Create the Final Answer Node

The final node generates an answer from the most recently retrieved context.

In [ ]:
def final_answer(state: AgenticRAGState):
    prompt = f"""
Answer using only this context.

QUESTION:
{state['question']}

CONTEXT:
{state.get('context', '')}
"""

    return {
        "answer": model.invoke(
            prompt
        )
    }

## Step 75 — Build the Agentic RAG Graph

The loop is created by this connection:

```text
rewrite
   ↓
retrieve
```

Complete flow:

```text
START
  ↓
retrieve
  ↓
check
  ↓
decision
  ├── generate → END
  └── rewrite
         ↓
      retrieve
```

In [ ]:
agentic_builder = StateGraph(
    AgenticRAGState
)

agentic_builder.add_node(
    "retrieve",
    agentic_retrieve
)

agentic_builder.add_node(
    "check",
    check_evidence
)

agentic_builder.add_node(
    "rewrite",
    rewrite_query
)

agentic_builder.add_node(
    "generate",
    final_answer
)

agentic_builder.add_edge(
    START,
    "retrieve"
)

agentic_builder.add_edge(
    "retrieve",
    "check"
)

agentic_builder.add_conditional_edges(
    "check",
    decide_rag
)

agentic_builder.add_edge(
    "rewrite",
    "retrieve"
)

agentic_builder.add_edge(
    "generate",
    END
)

agentic_rag_graph = agentic_builder.compile()

## Step 76 — Run the Agentic RAG Graph

**What to inspect.**

- How many retrieval attempts were used?
- What evidence score was produced?
- Was the original query rewritten?
- What final answer was generated?

In [ ]:
question = "Can student percentage discounts be stacked?"

result = agentic_rag_graph.invoke({
    "question": question,
    "query": question,
    "attempts": 0
})

print("Attempts:", result.get("attempts"))
print("Score:", result.get("score"))
print("Final query:", result.get("query"))
print("\nAnswer:")
print(result.get("answer"))

## Step 77 — Stream Node Updates

**Goal.** Observe the graph while it is running.

`invoke()` returns the final state.

`stream()` lets us inspect updates as nodes execute.

This is especially helpful when debugging branches and loops.

In [ ]:
question = "When is standard shipping free?"

for event in agentic_rag_graph.stream({
    "question": question,
    "query": question,
    "attempts": 0
}):
    print(event)
    print("-" * 60)

# Final Summary

## LangChain

LangChain gave us reusable application capabilities:

```text
Model
Prompt
Parser
Python functions in pipelines
Documents
Chunking
Embeddings
Chroma
Retriever
RAG
Tools
```

## LangGraph

LangGraph gave us explicit workflow control:

```text
State
Nodes
Edges
Conditional routing
Loops
Stopping conditions
LLM nodes
Tool nodes
RAG nodes
Agentic RAG
```

## The key distinction

```text
LangChain
What capabilities can the application use?

LangGraph
How should those capabilities execute?
```

A useful development pattern is:

```text
1. Build and test small functions.
2. Build and test LangChain components.
3. Put reliable components inside LangGraph nodes.
4. Add routing only when needed.
5. Add loops only when needed.
6. Always add a stopping condition to loops.
```

# Practice Exercises

Try the following in order:

1. Create a `divide_numbers()` tool and handle division by zero.
2. Create a break-even calculator tool.
3. Add a `discount` route to the LangGraph router.
4. Add another document to the RAG knowledge base.
5. Change the retriever from `k=2` to `k=3` and compare results.
6. Increase the Agentic RAG retry limit from 2 to 3.
7. Add a state field named `route_history` and record which path was selected.
8. Replace the simple word-based router with an LLM-generated route.